# 🚨 Exception Handling in Python 

---
## 📖 The Story: Why Do We Care?
Imagine you are writing a Data Engineering pipeline that processes **1,000 CSV files**. 
If file **500** is missing, an unhandled error will **crash the entire program**. Files **501-1000** will never be processed, and your company loses data. 

**Exception handling allows your program to:**
1. Anticipate the unexpected.
2. Handle it gracefully on the fly.
3. Prevent sudden crashes.
4. Keep the pipeline running.

## 🚦 The 2 Stages Where Errors Occur

```mermaid
graph TD
    A[Python Code] --> B{Parser Check}
    B -- "Grammar Violation" --> C[❌ Syntax Error]
    B -- "Grammar OK" --> D[Execution / Runtime]
    D --> E{Unforeseen Event?}
    E -- "Yes (e.g., Div by 0)" --> F[❌ Exception Raised]
    E -- "No" --> G[✅ Program Completes]
    F --> H{Handled via Try-Except?}
    H -- "Yes" --> I[🛡️ Program Continues]
    H -- "No" --> J[💥 Program Crashes]

### 🧠 Memory Trick: Error vs Exception
| Feature | Syntax Error | Exception (Runtime) |
| :--- | :--- | :--- |
| **When?** | Before code runs (Parsing) | While code is running |
| **Why?** | Bad grammar (missing `:`) | Bad logic/data (div by 0) |
| **Fix?** | Edit the code | Handle with `try-except` |


In [1]:
# ==========================================
# ❌ SYNTAX ERRORS (Prevent code from running)
# ==========================================
# Note: Uncomment these one by one. Colab will refuse to run the cell.

# 1. Missing colon
# if True
#     print("Missing colon")

# 2. Missing parenthesis
# print("Hello World"

# 3. Misspelled keyword
# impor math

# 4. Empty block (Needs 'pass')
# def my_func():

## 🌳 The Complete Exception Hierarchy
Most tutorials just say "inherit from `Exception`". But in interviews, you need to know the *actual* tree.

```text
BaseException (The Mother of All Exceptions)
 ├── SystemExit (Raised by sys.exit())
 ├── KeyboardInterrupt (Raised by Ctrl+C)
 ├── GeneratorExit (Raised by close() on generators)
 └── Exception (🌟 Base class for ALL regular errors)
      ├── ArithmeticError
      │    ├── ZeroDivisionError
      │    └── OverflowError
      ├── LookupError
      │    ├── IndexError (Lists/Tuples)
      │    └── KeyError (Dictionaries)
      ├── ValueError (Right type, wrong value)
      ├── TypeError (Wrong type)
      ├── OSError
      │    ├── FileNotFoundError
      │    └── PermissionError
      └── [Your Custom Exceptions Go Here!]
```

> 💡 **Golden Rule:** Never catch `BaseException` directly! It catches `KeyboardInterrupt` (Ctrl+C), making your program impossible to stop! Always catch `Exception`.


## 🔬 Deep Dive: The Exception Object (`e`)
When you write `except Exception as e:`, what exactly is `e`? It's a Python object with rich metadata.

*   **`type(e)`**: The class of the exception.
*   **`str(e)`**: The human-readable error message.
*   **`repr(e)`**: The developer-readable string (includes class name).
*   **`e.args`**: The tuple of arguments passed to the exception.
*   **`e.__traceback__`**: The stack trace object (useful for advanced logging).

In [2]:
# ==========================================
# 🔬 INSPECTING THE EXCEPTION OBJECT
# ==========================================
try:
    result = 10 / 0
except Exception as e:
    print(f"1. type(e)   : {type(e)}")
    print(f"2. str(e)    : {str(e)}")
    print(f"3. repr(e)   : {repr(e)}")
    print(f"4. e.args    : {e.args}")
    print(f"5. Traceback : {e.__traceback__}")

1. type(e)   : <class 'ZeroDivisionError'>
2. str(e)    : division by zero
3. repr(e)   : ZeroDivisionError('division by zero')
4. e.args    : ('division by zero',)
5. Traceback : <traceback object at 0x00000206D98EDC00>


## 🛡️ The Complete Exception Handling Structure

```mermaid
flowchart TD
    A[try: Risky Code] --> B{Exception Occurred?}
    B -- "Yes" --> C[except: Handle Error]
    B -- "No" --> D[else: Success Code]
    C --> E[finally: Cleanup Code]
    D --> E
    E --> F[Program Continues]
```

```
try     → "Let's try this risky thing..."
except  → "Oops, it failed. Handle it."
else    → "Wow, it worked! Do the success stuff."
finally → "No matter what, clean up the mess."
```

In [3]:
# ==========================================
# 📂 TRY-EXCEPT-ELSE-FINALLY IN ACTION
# ==========================================
def process_important_file(filepath):
    file = None
    try:
        print(f"🔵 TRY: Opening {filepath}...")
        file = open(filepath, 'r')
        data = file.read()
        # Simulating a runtime error
        result = 10 / 0  
    except ZeroDivisionError:
        print("🔴 EXCEPT: Math Error: Divided by zero!")
    except FileNotFoundError:
        print("🔴 EXCEPT: File not found!")
    else:
        print("🟢 ELSE: File read successfully!")
    finally:
        print("🟡 FINALLY: Cleaning up resources...")
        if file:
            file.close()
            print("🟡 FINALLY: File closed safely.\n")

# Test 1: File exists but math error
with open("dummy.txt", "w") as f: f.write("Data")
process_important_file("dummy.txt")

# Test 2: File doesn't exist
process_important_file("ghost.txt")

🔵 TRY: Opening dummy.txt...
🔴 EXCEPT: Math Error: Divided by zero!
🟡 FINALLY: Cleaning up resources...
🟡 FINALLY: File closed safely.

🔵 TRY: Opening ghost.txt...
🔴 EXCEPT: File not found!
🟡 FINALLY: Cleaning up resources...


## 🐍 Pythonic Philosophy: EAFP vs LBYL
This is a **massive** interview topic. How do you check for valid data?

### LBYL: "Look Before You Leap" (C / Java Style)
Check all conditions first. If valid, proceed.
*Problem:* Race conditions, verbose, slow.

### EAFP: "Easier to Ask Forgiveness than Permission" (Pythonic 🐍)
Just do it. If it fails, catch the exception.
*Benefit:* Cleaner, faster, avoids race conditions.

In [4]:
# ==========================================
# 🐍 EAFP vs LBYL
# ==========================================
data = {'name': 'Anuj'}

# ❌ LBYL (Look Before You Leap)
if 'salary' in data and isinstance(data['salary'], int):
    salary = data['salary']
else:
    salary = 0

# ✅ EAFP (Easier to Ask Forgiveness than Permission)
try:
    salary = data['salary']
except (KeyError, TypeError):
    salary = 0
    
print(f"Salary using EAFP: {salary}")

Salary using EAFP: 0


## 🛑 Assertions (`assert`) vs Exceptions (`raise`)
Beginners confuse these. Here is the rule:

*   **`assert`**: Used for **internal sanity checks** during development. It should *never* fail in production. (Can be disabled with `python -O`).
*   **`raise`**: Used for **expected runtime errors** and business logic validation. Always active.

In [5]:
# ==========================================
# 🛑 ASSERTIONS VS EXCEPTIONS
# ==========================================
def calculate_discount(price, discount):
    # Internal sanity check (Developer error if this fails)
    assert price > 0, "Price must be positive"
    
    # Business rule validation (User error if this fails)
    if discount > 100:
        raise ValueError("Discount cannot exceed 100%")
        
    return price - (price * discount / 100)

try:
    calculate_discount(100, 150)
except ValueError as e:
    print(f"Caught Business Error: {e}")

Caught Business Error: Discount cannot exceed 100%


## 🔗 Exception Chaining (`raise ... from ...`)
In production, you often catch a low-level error and raise a high-level domain error. 
Using `from e` preserves the **original traceback**, which is a lifesaver for debugging!

In [6]:
# ==========================================
# 🔗 EXCEPTION CHAINING
# ==========================================
def fetch_user_data(user_id):
    try:
        # Simulating a database failure
        int("not_a_number") 
    except ValueError as e:
        # Chain the exception to preserve the original cause!
        raise RuntimeError(f"Failed to fetch user {user_id}") from e

try:
    fetch_user_data(101)
except RuntimeError as e:
    print(f"Caught: {e}")
    print(f"Original Cause: {e.__cause__}")

Caught: Failed to fetch user 101
Original Cause: invalid literal for int() with base 10: 'not_a_number'


## 🏗️ Designing Custom Exception Hierarchies
Don't just create random exceptions. Create a **Base Class** for your app, then subclass it. This allows catching broad or specific errors easily.

In [7]:
# ==========================================
# 🏗️ CUSTOM EXCEPTION HIERARCHY
# ==========================================
class BankError(Exception):
    """Base exception for all banking operations"""
    pass

class InsufficientFundsError(BankError):
    """Raised when withdrawal exceeds balance"""
    pass

class InvalidPinError(BankError):
    """Raised when PIN is incorrect"""
    pass

class AccountLockedError(BankError):
    """Raised when account is locked due to too many attempts"""
    pass

def process_withdrawal(pin, amount, balance):
    if pin != "1234":
        raise InvalidPinError("Incorrect PIN")
    if amount > balance:
        raise InsufficientFundsError(f"Need ${amount - balance} more")
    return balance - amount

try:
    process_withdrawal("9999", 500, 1000)
except InvalidPinError:
    print("🔐 Security: Invalid PIN")
except BankError as e: # Catches InsufficientFundsError too!
    print(f"🏦 Bank Error: {e}")

🔐 Security: Invalid PIN


## 📊 Production Logging & Data Science
In production, we **never** use `print(e)`. We use the `logging` module. 
Also, Data Science libraries have their own specific exceptions!

In [8]:
# ==========================================
# 📊 LOGGING & DATA SCIENCE EXCEPTIONS
# ==========================================
import logging

# Configure logging for production
logging.basicConfig(
    level=logging.ERROR, 
    format='%(asctime)s | %(levelname)s | %(message)s'
)

# 1. Logging instead of printing
try:
    10 / 0
except ZeroDivisionError as e:
    # exc_info=True automatically attaches the stack trace!
    logging.error(f"Math operation failed", exc_info=True)

# 2. Pandas Exception Handling
import pandas as pd
try:
    # Simulating a bad CSV
    df = pd.read_csv("non_existent_data.csv")
except FileNotFoundError as e:
    logging.warning(f"Data file missing: {e}")
except pd.errors.ParserError as e:
    logging.error(f"CSV parsing failed: {e}")

# 3. API / Requests Exception Handling
import requests
try:
    response = requests.get("https://api.invalid-url-12345.com", timeout=2)
except requests.exceptions.ConnectionError:
    logging.error("Failed to connect to API")
except requests.exceptions.Timeout:
    logging.error("API request timed out")

2026-06-19 23:22:21,571 | ERROR | Math operation failed
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_24464\1274369070.py", line 14, in <module>
    10 / 0
    ~~~^~~
ZeroDivisionError: division by zero
2026-06-19 23:22:25,007 | ERROR | Failed to connect to API


## 🏦 Production Mini-Project: The ATM System
Let's combine **everything** we've learned into a real-world scenario:
1. Custom Exception Hierarchy
2. `try-except-else-finally`
3. Exception Chaining
4. Production Logging
5. Business Rule Validation

In [9]:
# ==========================================
# 🏦 PRODUCTION ATM SYSTEM
# ==========================================
import logging

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# 1. Custom Hierarchy
class ATMError(Exception): pass
class InvalidCardError(ATMError): pass
class InsufficientFundsError(ATMError): pass

# 2. The ATM Class
class ATM:
    def __init__(self, card_number, pin, balance):
        self.card_number = card_number
        self.pin = pin
        self.balance = balance
        self.is_logged_in = False

    def authenticate(self, input_pin):
        try:
            if input_pin != self.pin:
                raise InvalidCardError("Authentication failed: Wrong PIN")
            self.is_logged_in = True
            logging.info("✅ Authentication successful")
        except InvalidCardError as e:
            logging.error(f"🔐 {e}")
            raise

    def withdraw(self, amount):
        if not self.is_logged_in:
            raise ATMError("Must authenticate first")
        
        try:
            if amount > self.balance:
                shortfall = amount - self.balance
                raise InsufficientFundsError(
                    f"Requested ${amount}, Available ${self.balance}"
                ) from None # 'from None' hides the internal traceback
            self.balance -= amount
            logging.info(f"💵 Dispensed ${amount}. New balance: ${self.balance}")
        except InsufficientFundsError as e:
            logging.error(f"❌ {e}")
            raise
        finally:
            # ALWAYS run cleanup
            self.is_logged_in = False
            logging.info("🔒 Session closed. Card ejected.\n")

# 3. Execution
atm = ATM("1234-5678", "9999", 1000)

print("--- Scenario 1: Successful Withdrawal ---")
try:
    atm.authenticate("9999")
    atm.withdraw(400)
except ATMError:
    pass

print("--- Scenario 2: Insufficient Funds ---")
try:
    atm.authenticate("9999")
    atm.withdraw(2000)
except ATMError:
    pass

2026-06-19 23:23:07,065 | ERROR | ❌ Requested $2000, Available $600


--- Scenario 1: Successful Withdrawal ---
--- Scenario 2: Insufficient Funds ---


## 🗺️ The Master Visual Summary
Save this diagram! It connects every concept in Exception Handling.

```mermaid
flowchart TD
    A[Syntax Error: Grammar] -->|Fix Code| B[Runtime Error: Logic]
    B --> C[Exception Raised]
    C --> D{try block}
    D -->|Error| E[except block]
    D -->|Success| F[else block]
    E --> G[finally block: Cleanup]
    F --> G
    G --> H{Need Custom Logic?}
    H -->|Yes| I[raise Custom Exception]
    H -->|No| J[Program Continues]
    I --> K[Exception Hierarchy]
    K --> L[Log with exc_info=True]
```

## 📋 The Ultimate Exception Handling Cheat Sheet

| Concept | Syntax / Rule |
| :--- | :--- |
| **Catch Specific** | `except ValueError as e:` (Always do this!) |
| **Catch Multiple** | `except (TypeError, ValueError) as e:` |
| **Success Path** | `else:` (Runs only if `try` succeeds) |
| **Cleanup Path** | `finally:` (Runs ALWAYS, even on `return`) |
| **Manual Raise** | `raise ValueError("Bad input")` |
| **Chain Errors** | `raise RuntimeError("Fail") from e` |
| **Log Errors** | `logging.error("Msg", exc_info=True)` |
| **Custom Base** | `class MyAppError(Exception): pass` |

---
## 🎤 Master Interview Questions

**Q1: What is the difference between `else` and `finally`?**
> `else` runs ONLY if no exception occurred. `finally` runs ALWAYS, regardless of success or failure.

**Q2: Why is `except:` (bare except) dangerous?**
> It catches `BaseException`, including `SystemExit` and `KeyboardInterrupt` (Ctrl+C). You trap the user and can't kill the script!

**Q3: Explain EAFP vs LBYL.**
> EAFP (Pythonic) tries the action and catches the error. LBYL (C-style) checks conditions first. EAFP avoids race conditions and is cleaner.

**Q4: What does `raise ... from e` do?**
> It chains exceptions, preserving the original traceback (`__cause__`) for debugging while raising a higher-level domain error.

**Q5: How do you handle exceptions in Pandas/Scikit-Learn?**
> Catch specific library errors like `pd.errors.ParserError` or `ValueError` for shape mismatches, and always log them using `logging`.

## 💡 Fun Facts
1. 🚀 **Mars Climate Orbiter**: NASA lost a $327M spacecraft because of a unit conversion error. A proper exception/validation system could have caught it!
2. 🎣 **EAFP Origin**: "Easier to ask for forgiveness than permission" is a famous Grace Hopper quote, adopted as Python's core philosophy.
3. 🏗️ **Google's Rule**: Google's internal Python style guide strictly forbids bare `except:` clauses and mandates logging `exc_info=True`.

---
## 🎉 Congratulations!
You have completed the **Elite Exception Handling Masterclass**.
You now know how to build robust, production-grade Python applications.

### 🚀 What's Next?
Now that you can handle errors, how do you organize thousands of lines of code?
**Next Topic:** Modules, Packages, and Virtual Environments! 📦